# Results analysis
Analysis only: reads `reports/tables/` and `runs/`. **No training happens here.** Regenerate the inputs with `make report c4c5 c6`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json, glob
from pathlib import Path
T = Path('../reports/tables')
res = pd.read_csv(T / 'results.csv'); res

## C2: accuracy vs. CPU latency
Each point is one architecture variant from the sweep. Trained variants carry their test AUROC.

In [ ]:
sweep = pd.read_csv(T / 'c2_latency_sweep.csv')
fig, ax = plt.subplots(figsize=(6, 4))
for arch, g in sweep.groupby('arch'):
    ax.scatter(g['p95_ms'], g['n_params'] / 1e6, label=arch)
ax.axvline(20, ls='--', c='k', lw=1); ax.set(xscale='log', xlabel='p95 latency (ms, 4 threads)', ylabel='params (M)'); ax.legend()

## C3: data efficiency

In [ ]:
rows = []
for f in glob.glob('../runs/c[13]_*/seed*/metrics.json'):
    m = json.load(open(f))
    if m['name'].startswith('c1_') and m['name'].count('_') == 1 or m['name'].startswith('c3_'):
        frac = float(m['name'].split('frac')[1]) if 'frac' in m['name'] else 1.0
        rows.append({'model': m['model'], 'fraction': frac, 'auroc': m['test']['macro_auroc']})
c3 = pd.DataFrame(rows).groupby(['model', 'fraction'])['auroc'].agg(['mean', 'std']).reset_index()
fig, ax = plt.subplots(figsize=(6, 4))
for model, g in c3.groupby('model'):
    ax.errorbar(g['fraction'], g['mean'], yerr=g['std'], marker='o', capsize=3, label=model)
ax.set(xscale='log', xlabel='fraction of folds 1-8', ylabel='test macro AUROC'); ax.legend(); c3

## C4/C5: quality gate

In [ ]:
display(pd.read_csv(T / 'c4_gate_auroc.csv'))
print((T / 'c5_rejection.md').read_text())

## C6: robustness

In [ ]:
c6 = pd.read_csv(T / 'c6_robustness.csv')
c6[c6.name != 'quality_gate'].groupby(['corruption', 'level', 'name'])['macro_auroc'].mean().unstack('name')